#Gemma 4 Developer Agent

## 1. Environment Configuration and Package Installation

In [ ]:
import glob
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

assert WHEELHOUSE_DIR.exists(), (
    f'Wheelhouse not found at {WHEELHOUSE_DIR}. '
    'Attach the wheelhouse dataset in the Input panel (check !ls /kaggle/input/datasets).'
)

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        os.symlink(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
print('Wheelhouse installation complete.')

## 2. Competition Dataset and Sample Submission

In [ ]:
# --- Guard: the install cell must have run first ---
try:
    import swegemma
except ModuleNotFoundError:
    raise RuntimeError(
        "swegemma is not installed. Run the 'Environment Configuration and Package "
        "Installation' cell first, wait for 'Wheelhouse installation complete.', "
        "then rerun this cell."
    )

from swegemma.models import load_tasks
import zipfile
import shutil
from pathlib import Path

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# --- Locate YOUR submission dataset (the attached dataset that contains agent.yaml) ---
# If auto-detection picks the wrong folder, set it by hand, e.g.:
# MY_SUBMISSION_SRC = Path('/kaggle/input/datasets/<owner>/<slug>')
MY_SUBMISSION_SRC = None

if MY_SUBMISSION_SRC is None:
    INPUT_ROOT = Path('/kaggle/input')
    patterns = [
        'datasets/*/*/agent.yaml',
        'datasets/*/*/*/agent.yaml',   # in case it sits inside a wrapper folder
        '*/agent.yaml',
        '*/*/agent.yaml',
    ]
    hits = []
    for pat in patterns:
        hits += [
            p for p in INPUT_ROOT.glob(pat)
            if 'competitions' not in p.parts and 'sample_submission' not in p.parts
        ]

    if hits:
        MY_SUBMISSION_SRC = hits[0].parent
    else:
        # Fallback: the dataset may have kept your files as a .zip
        for z in INPUT_ROOT.glob('datasets/*/*/*.zip'):
            with zipfile.ZipFile(z) as zf:
                if any(n.endswith('agent.yaml') for n in zf.namelist()):
                    extract_dir = Path('/tmp/my_submission_src')
                    shutil.rmtree(extract_dir, ignore_errors=True)
                    zf.extractall(extract_dir)
                    found = list(extract_dir.rglob('agent.yaml'))
                    MY_SUBMISSION_SRC = found[0].parent
                    break

if MY_SUBMISSION_SRC is None:
    listing = sorted(str(p) for p in Path('/kaggle/input').glob('*/*')) + \
              sorted(str(p) for p in Path('/kaggle/input').glob('datasets/*/*'))
    raise FileNotFoundError(
        'Could not find agent.yaml in any attached dataset. '
        'Attach your submission dataset in the Input panel. Found:\n' + '\n'.join(listing)
    )

print(f'Using submission source: {MY_SUBMISSION_SRC}')

# Copy YOUR submission into a writable directory (folder name kept so other cells still work)
AGENT_DIR = WORKING_DIR / 'sample_submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
shutil.copytree(MY_SUBMISSION_SRC, AGENT_DIR)

# Keep stray metadata out of the scored zip
(AGENT_DIR / 'dataset-metadata.json').unlink(missing_ok=True)

# Sanity checks
assert (AGENT_DIR / 'agent.yaml').exists(), 'agent.yaml is not at the top level of the submission'
assert (AGENT_DIR / 'configs' / 'sampling.yaml').exists(), 'configs/sampling.yaml is missing'

# NOTE: the old code that overwrote configs/sampling.yaml has been removed on purpose.
# Your repo's configs/sampling.yaml is now the source of truth.

# Load tasks from the published dataset
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)

print(f'Data directory: {DATA_DIR}')
print(f'Agent directory: {AGENT_DIR}')
print(f'Loaded {len(tasks)} tasks from {TASKS_PATH.name}')
for t in tasks[:5]:
    print(f'  - {t.instance_id} ({t.repo} @ {t.base_commit[:8]})')

## TESTING. Check for config file

In [ ]:
"""
print('source dataset folder:', MY_SUBMISSION_SRC)
print('\n--- eval_config.yaml in the dataset folder ---')
print((Path(MY_SUBMISSION_SRC) / 'eval_config.yaml').read_text())
print('--- eval_config.yaml in AGENT_DIR (what the run used) ---')
print((AGENT_DIR / 'eval_config.yaml').read_text())
print('--- values cell 12 parsed ---')
print(timeout_seconds, max_tool_calls, max_time_minutes, max_turns)
print('--- limits ---')
print(limits)
print(gen_constraints)
"""

## TESTING. Print the file tree

In [ ]:
from pathlib import Path

root = Path('/kaggle/working/sample_submission')
assert root.exists(), 'Folder not found. Run the install cell and then the data cell first.'

TEXT_EXT = {'.yaml', '.yml', '.md', '.txt', '.json', '.py', '.toml'}

# File tree with sizes
for p in sorted(root.rglob('*')):
    depth = len(p.relative_to(root).parts) - 1
    size = '' if p.is_dir() else f'  ({p.stat().st_size:,} bytes)'
    print('  ' * depth + ('📁 ' if p.is_dir() else '📄 ') + p.name + size)

# Contents of small text files (config, prompts, skills)
for p in sorted(root.rglob('*')):
    if p.is_file() and p.suffix in TEXT_EXT and p.stat().st_size < 20_000:
        print(f'\n{"=" * 20} {p.relative_to(root)} {"=" * 20}')
        print(p.read_text(errors='replace'))

## 3. Code Graph Functions

In [ ]:
from swegemma import graph as sg

sample_task = tasks[0]
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')

# Load the pre-computed repository code graph and node embeddings for the sample task
repo_graph = sg.get_graph(
    repo_name=sample_task.repo,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
sample_nodes = list(repo_graph.nodes())
print(f'Graph for {sample_task.repo} ({sample_task.instance_id}):')
print(f'  Nodes: {repo_graph.number_of_nodes()}, Edges: {repo_graph.number_of_edges()}')

# Select a connected symbol and query structural neighbors (callers, callees, definitions)
query_node = next((n for n, deg in repo_graph.degree() if deg >= 5), sample_nodes[0])
neighbors = sg.get_neighbor(
    node=query_node,
    graph=repo_graph,
    max_neighbors=10,
)
print(f'\nNeighbors of {query_node!r} (showing up to 5):')
for n in neighbors[:5]:
    print(f'  - {n}')

# Search for semantically similar code nodes using pre-computed vector embeddings
similar = sg.get_similar_nodes(
    node=query_node,
    repo_name=sample_task.repo,
    k=5,
    graph=repo_graph,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
print(f'\nTop similar nodes to {query_node!r}:')
for item in similar:
    print(f"  - {item['node_name']} (similarity={item['similarity']:.4f})")

# Extract an induced subgraph connecting the query symbol and its neighbors
focal_nodes = [query_node, *neighbors[:4]]
subgraph = sg.get_induced_subgraph(repo_graph, focal_nodes)
print(
    f'\nInduced subgraph over {len(focal_nodes)} focal nodes: '
    f'{subgraph.number_of_nodes()} nodes, {subgraph.number_of_edges()} edges'
)

## 4. Start vLLM Server

In [ ]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
from swegemma.models.discovery import validate_single_declared_model

litellm.drop_params = True

TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
INFERENCE_API_KEY = 'EMPTY'

# Validate single base model and discover any PEFT LoRA adapters in the submission directory
declared_model = validate_single_declared_model(AGENT_DIR)
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=True,
    max_loras=8,
    max_lora_rank=128,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

# Register model and LoRA adapter aliases for Google ADK
models = server_instance.create_model_registry(
    aliases=[declared_model, TARGET_MODEL_NAME],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)

## 5. Run Phase 1 Inference and Phase 2 Verification

In [ ]:
import asyncio
import concurrent.futures
import pandas as pd
import yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator


def run_sync(coro_or_fn, *args, **kwargs):
    """Run an async coroutine synchronously inside a notebook event loop."""
    fn = (lambda: coro_or_fn(*args, **kwargs)) if callable(coro_or_fn) else (lambda: coro_or_fn)
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())


# Read evaluation settings from the submission's eval_config.yaml
eval_config_file = AGENT_DIR / 'eval_config.yaml'
raw_eval_cfg = yaml.safe_load(eval_config_file.read_text(encoding='utf-8'))
eval_section = raw_eval_cfg.get('evaluation', raw_eval_cfg)

timeout_seconds = int(eval_section.get('timeout_seconds', 300))
max_tool_calls = int(eval_section.get('max_tool_calls', 100))
max_time_minutes = float(eval_section.get('max_time_minutes', 60.0))
turns_raw = eval_section.get('max_turns', eval_section.get('max_llm_calls'))
max_turns = int(turns_raw) if turns_raw is not None else None

# CHANGED: easy-to-edit task count (start at 10; use 2 for a quick smoke test)
NUM_TASKS = 2
SAMPLE_TASKS = tasks[:NUM_TASKS]
limits, gen_constraints = build_submission_limits()

# CHANGED: show what is actually in effect, including any competition-enforced caps
print('eval_config.yaml ->', dict(timeout_seconds=timeout_seconds,
                                  max_tool_calls=max_tool_calls,
                                  max_time_minutes=max_time_minutes,
                                  max_turns=max_turns))
print('limits:', limits)
print('generation constraints:', gen_constraints)

# Configure Evaluator to run Phase 1 (agent patch generation) and Phase 2 (verification)
# using the subprocess sandbox backend and the submission's eval_config parameters
eval_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=WORKING_DIR / 'results',
    submission_dir=AGENT_DIR,
    models=models,
    sandbox='subprocess',
    timeout_seconds=timeout_seconds,
    max_time_minutes=max_time_minutes,
    max_tool_calls=max_tool_calls,
    max_turns=max_turns,
    limits=limits,
    generation_constraints=gen_constraints,
    adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15,
        overlap_size=2,
        token_threshold=14336,
        event_retention_size=5,
    ),
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    wheels_dir=DATA_DIR / 'wheels',
    verbose=False,
)

evaluator = Evaluator(eval_config)
predictions = []
run_log = []  # CHANGED: keep per-task stats for the summary

for idx, task in enumerate(SAMPLE_TASKS, start=1):
    print(f'[{idx}/{len(SAMPLE_TASKS)}] Evaluating {task.instance_id} ({task.repo})...')
    result = run_sync(
        evaluator.evaluate_task,
        task=task,
        task_index=idx,
        total_tasks=len(SAMPLE_TASKS),
    )
    predictions.append({'id': task.instance_id, 'prediction': result.agent_patch or ''})
    run_log.append({
        'id': task.instance_id,
        'repo': task.repo,
        'resolved': bool(result.resolved),
        'patch_chars': len(result.agent_patch or ''),
        'tool_calls': result.tool_calls,
        'duration_s': round(result.duration_seconds, 1),
    })
    print(
        f'  -> resolved={result.resolved}, '
        f'exit_code={result.test_exit_code}, '
        f'patch_chars={len(result.agent_patch or "")}, '
        f'tool_calls={result.tool_calls}, '
        f'duration={result.duration_seconds:.1f}s'
    )

submission_df = pd.DataFrame(predictions, columns=['id', 'prediction'])

# CHANGED: summary table to paste into notes/experiments.md
summary_df = pd.DataFrame(run_log)
n_resolved = int(summary_df['resolved'].sum())
n_patched = int((summary_df['patch_chars'] > 0).sum())
print(f'\nResolved {n_resolved}/{len(summary_df)} | '
      f'non-empty patches {n_patched}/{len(summary_df)} | '
      f'avg tool calls {summary_df["tool_calls"].mean():.1f} | '
      f'avg duration {summary_df["duration_s"].mean():.1f}s')
display(summary_df)

## 6. Package Submission Archive

In [ ]:
import zipfile
from swegemma.config import ALLOWED_SUBMISSION_EXTENSIONS, MAX_SUBMISSION_SIZE_BYTES

# Create submission.zip with agent.yaml at the root of the archive
zip_base = WORKING_DIR / 'submission'
zip_path = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=AGENT_DIR))

# Validate archive contents against competition constraints
with zipfile.ZipFile(zip_path, 'r') as zf:
    infos = zf.infolist()
    total_size = sum(i.file_size for i in infos)
    assert total_size <= MAX_SUBMISSION_SIZE_BYTES, f'Archive exceeds 3 GiB limit: {total_size}'
    for info in infos:
        if not info.is_dir():
            ext = Path(info.filename).suffix.lower()
            assert ext in ALLOWED_SUBMISSION_EXTENSIONS, f'Disallowed file extension: {info.filename}'

print(f'Created {zip_path} ({zip_path.stat().st_size / (1024 * 1024):.2f} MiB)')